In [1]:
!nvidia-smi

Tue Sep  1 16:29:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
base = "/content/drive/MyDrive/player detection"
print(os.listdir(base))

['README.roboflow.txt', 'README.dataset.txt', 'data.yaml', 'train', 'valid', 'training', 'videos', 'tracking']


In [8]:
# Hn3ml validation
images_path = f"{base}/train/images"
labels_path = f"{base}/train/labels"

images = [
    f for f in os.listdir(images_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]

labels = [
    f for f in os.listdir(labels_path)
    if f.endswith(".txt")
]

print("Images:", len(images))
print("Labels:", len(labels))

Images: 480
Labels: 480


In [6]:
class_counts = {0: 0, 1: 0, 2: 0}
bad_labels = []

for filename in labels:
    filepath = os.path.join(labels_path, filename)

    with open(filepath, "r") as f:
        for line_number, line in enumerate(f, start=1):

            parts = line.strip().split()

            if len(parts) != 5:
                bad_labels.append(
                    f"{filename} line {line_number}: wrong format"
                )
                continue

            try:
                class_id = int(parts[0])
                values = [float(x) for x in parts[1:]]
            except ValueError:
                bad_labels.append(
                    f"{filename} line {line_number}: non-numeric value"
                )
                continue

            if class_id not in class_counts:
                bad_labels.append(
                    f"{filename} line {line_number}: invalid class {class_id}"
                )
            else:
                class_counts[class_id] += 1

            if not all(0 <= x <= 1 for x in values):
                bad_labels.append(
                    f"{filename} line {line_number}: invalid bbox values"
                )

print("GoalKeeper:", class_counts[0])
print("Player:", class_counts[1])
print("referee:", class_counts[2])
print("Bad labels:", len(bad_labels))

GoalKeeper: 268
Player: 7219
referee: 433
Bad labels: 0


In [ ]:
import random
import shutil

valid_images = f"{base}/valid/images"
valid_labels = f"{base}/valid/labels"

os.makedirs(valid_images, exist_ok=True)
os.makedirs(valid_labels, exist_ok=True)

random.seed(42)
random.shuffle(images)

val_count = int(len(images) * 0.20)
val_images = images[:val_count]

for image in val_images:

    label_name = os.path.splitext(image)[0] + ".txt"

    shutil.move(
        os.path.join(images_path, image),
        os.path.join(valid_images, image)
    )

    shutil.move(
        os.path.join(labels_path, label_name),
        os.path.join(valid_labels, label_name)
    )

print("Train images:", len(os.listdir(images_path)))
print("Train labels:", len(os.listdir(labels_path)))

print("Valid images:", len(os.listdir(valid_images)))
print("Valid labels:", len(os.listdir(valid_labels)))

In [ ]:
yaml_content = f"""
train: {base}/train/images
val: {base}/valid/images

nc: 3

names:
  0: GoalKeeper
  1: Player
  2: referee
"""

with open("/content/data.yaml", "w") as f:
    f.write(yaml_content)

print(open("/content/data.yaml").read())

In [ ]:
!pip install -q ultralytics

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset("/content/data.yaml")

print("Dataset checked successfully!")
print("Train:", data["train"])
print("Val:", data["val"])
print("Classes:", data["names"])

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    project="/content/drive/MyDrive/player detection/training",
    name="football_yolo"
)

In [ ]:
best_model = "/content/drive/MyDrive/player detection/training/football_yolo/weights/best.pt"

print("Model exists:", os.path.exists(best_model))

In [ ]:
videos_path = "/content/drive/MyDrive/player detection/videos"

os.makedirs(videos_path, exist_ok=True)

print("Videos folder ready!")

In [ ]:
print(os.listdir(videos_path))

In [ ]:
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/player detection/training/football_yolo/weights/best.pt"

VIDEO_PATH = "/content/drive/MyDrive/player detection/videos/match.mp4"

model = YOLO(MODEL_PATH)

results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    conf=0.25,
    iou=0.5,
    persist=True,
    save=True,
    project="/content/drive/MyDrive/player detection/tracking",
    name="match_tracking"
)

print("Tracking completed!")

In [ ]:
tracking_path = "/content/drive/MyDrive/player detection/tracking"

print(os.listdir(tracking_path))

In [ ]:
from IPython.display import Video, display

output_video = "/content/drive/MyDrive/player detection/tracking/match_tracking/match.avi"

display(Video(output_video, embed=True))